# Know Your IP

Enrich IP addresses with geolocation, registry, reputation, and exposure data.

This notebook runs top to bottom **with no API keys and no databases**. The
keyless providers come first for that reason; everything needing an account is
guarded by its `enabled` flag, so those cells are skipped rather than failing.

In [ ]:
from pprint import pprint

from know_your_ip import load_config, query_ip

config = load_config()
IP = "8.8.8.8"

## What is registered

Each provider declares what it costs, whether it needs a key, and whether it
can answer a question about the past.

In [ ]:
from know_your_ip.providers import REGISTRY

for p in REGISTRY.providers:
    key = "key required" if p.requires_key else "no key"
    print(f"{p.name:<12} {p.cost:<9} {key:<13} enabled={p.is_enabled(config)}")

## The keyless path

`network` classifies the address offline and resolves its PTR record. `rdap`
asks the responsible registry - the IETF successor to port-43 WHOIS - via
IANA's bootstrap registry. Neither needs an account.

In [ ]:
pprint(query_ip(config, IP, providers=["network", "rdap"]))

Classification is not just description. Addresses that are not globally
routable cannot be looked up by any external service, so knowing that up front
saves metered quota on data that is full of RFC1918.

In [ ]:
for address in ["8.8.8.8", "192.168.1.1", "100.64.0.1", "2001:4860:4860::8888"]:
    record = query_ip(config, address, providers=["network"])
    print(
        f"{address:<24} {record['network.category']:<10} "
        f"routable={record['network.is_routable']}"
    )

`100.64.0.1` is carrier-grade NAT: shared by many subscribers, so anything
inferred about an individual behind it is much weaker than for a normal public
address.

## Services that need an account

Each is skipped unless enabled in your configuration. See
`examples/know_your_ip.toml` for the full file, or set environment variables
such as `KNOW_YOUR_IP_VIRUSTOTAL_API_KEY`.

In [ ]:
from know_your_ip import (
    abuseipdb_api,
    apivoid_api,
    censys_api,
    maxmind_geocode_ip,
    shodan_api,
    virustotal_api,
)

for label, enabled, fn in [
    ("MaxMind", config.maxmind.enabled, maxmind_geocode_ip),
    ("AbuseIPDB", config.abuseipdb.enabled, abuseipdb_api),
    ("VirusTotal", config.virustotal.enabled, virustotal_api),
    ("Censys", config.censys.enabled, censys_api),
    ("Shodan", config.shodan.enabled, shodan_api),
    ("APIVoid", config.apivoid.enabled, apivoid_api),
]:
    if not enabled:
        print(f"{label}: not enabled, skipping")
        continue
    print(f"\n--- {label} ---")
    try:
        pprint(fn(config, IP))
    except Exception as exc:
        print(f"{label} failed: {exc}")

MaxMind needs a database, which needs a free account. Fetch it with:

```
know_your_ip download-db --account-id YOUR_ID --license-key YOUR_KEY
```

It lands in the cache directory where the package finds it with no further
configuration.

## Everything at once

`query_ip` runs every enabled provider and returns the **complete** record. The
`[output]` column list only controls what the command line writes to CSV.

In [ ]:
record = query_ip(config, IP)
print(f"{len(record)} fields collected")
pprint(record)

## Caching, and why it is append-only

Results are cached on disk, so re-running costs no API quota - which is what
makes a 500/day free tier usable at research sample sizes.

Rows are **appended, not replaced**. Enrich the same address list monthly and
the cache accumulates a panel dataset rather than overwriting itself.

In [ ]:
from datetime import timedelta

from know_your_ip.cache import Cache

with Cache("example-cache.sqlite") as cache:
    # Three runs, forcing a refetch each time so the history is visible.
    for _ in range(3):
        query_ip(config, IP, providers=["network"], cache=cache, max_age=timedelta(0))

    for row in cache.history(IP, provider="network"):
        print(row["observed_at"], "->", row["data"]["network.category"])

    print("\nstats:", cache.stats())

## Asking about the past

`as_of` asks what was true on a date. No provider supports it yet - and that is
the point. Providers that cannot answer historically are **skipped with a
warning** rather than returning today's data under a historical label.

In [ ]:
from datetime import date

# Returns just the address: nothing can answer for 2019, so nothing pretends to.
print(query_ip(config, IP, as_of=date(2019, 3, 14)))

## Batch processing

From the command line, over a file:

```
know_your_ip --file input.csv --providers network,rdap --cache obs.sqlite -o out.csv
```

Invalid addresses are reported and skipped rather than aborting the run, and a
provider failing the same way for every address is logged once, not once per
row.

In [ ]:
import pandas as pd  # pip install 'know_your_ip[pandas]'

addresses = ["8.8.8.8", "1.1.1.1", "9.9.9.9"]
df = pd.DataFrame(query_ip(config, a, providers=["network", "rdap"]) for a in addresses)
df[["ip", "network.category", "network.reverse_dns", "rdap.name"]]